# Innovative Out-of-Stock Prediction System
**Reference**: Giaconia, C., & Chamas, A. (2023). *Innovative Out-of-Stock Prediction System Based on Data History Knowledge Deep Learning Processing.*

**Goal**: To replicate the paper's Deep Learning pipeline which uses a CNN backbone, a Multi-Head Self-Attention Booster, and a Temporal Convolutional Network (TCN) to estimate the hourly stockout probabilities over the next 24 hours.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Loading & Sequence Windowing

In [ ]:
def load_and_prep(path):
    df = pd.read_parquet(path)
    df['series_id'] = df['city_id'].astype(str) + '_' + df['store_id'].astype(str) + '_' + df['product_id'].astype(str)
    df = df.sort_values(['series_id', 'dt'])
    
    # Decode bytes to numpy arrays
    df['hours_sale_arr'] = df['hours_sale'].apply(lambda x: np.frombuffer(x, dtype=np.float64) if isinstance(x, bytes) else np.array(x))
    df['hours_stock_arr'] = df['hours_stock_status'].apply(lambda x: np.frombuffer(x, dtype=np.int32) if isinstance(x, bytes) else np.array(x))
    
    df['day_of_week'] = pd.to_datetime(df['dt']).dt.dayofweek
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)
    
    return df

print("Loading data...")
train_df = load_and_prep('../data/top15_train.parquet')
test_df = load_and_prep('../data/top15_test.parquet')

class SequenceDataset(Dataset):
    def __init__(self, df, seq_len=14):
        self.seq_len = seq_len
        self.X, self.Y = self._build_sequences(df)
        
    def _build_sequences(self, df):
        X, Y = [], []
        for series_id, group in tqdm(df.groupby('series_id')):
            group = group.reset_index(drop=True)
            if len(group) <= self.seq_len:
                continue
                
            for i in range(len(group) - self.seq_len):
                window = group.iloc[i : i + self.seq_len]
                target_day = group.iloc[i + self.seq_len]
                
                # F=34 Features per day
                features = []
                for _, row in window.iterrows():
                    day_feats = [
                        row['sale_amount'], 
                        row['stock_hour6_22_cnt'],
                        row['discount'], row['holiday_flag'], row['activity_flag'],
                        row['precpt'], row['avg_temperature'], row['avg_humidity'],
                        row['dow_sin'], row['dow_cos']
                    ]
                    day_feats.extend(row['hours_sale_arr'][:24].tolist())
                    features.append(day_feats)
                
                target_status = target_day['hours_stock_arr'][:24]
                # Invert so 1 = stockout, 0 = stocked (for positive class prediction)
                target = 1.0 - target_status
                
                X.append(features)
                Y.append(target.tolist())
                
        return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Transpose X to (Channels, Seq_Len) for Conv1D
        return self.X[idx].transpose(0, 1), self.Y[idx]

print("Building train sequences...")
train_ds = SequenceDataset(train_df)
print("Building test sequences...")
test_ds = SequenceDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
print(f"Train sequences: {len(train_ds)}, Test sequences: {len(test_ds)}")

## 2. Model Architecture: CNN + Attention + TCN

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        padding = (3 - 1) * dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(0.1)
        self.padding = padding
        
        self.shortcut = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.shortcut(x)
        x = self.dropout(self.gelu(self.bn1(self.conv1(x)[:, :, :-self.padding])))
        x = self.dropout(self.gelu(self.bn2(self.conv2(x)[:, :, :-self.padding])))
        return x + res

class GiaconiaOOSModel(nn.Module):
    def __init__(self, in_features=34, seq_len=14, out_classes=24):
        super().__init__()
        # Stage 1: CNN
        self.cnn = nn.Sequential(
            nn.Conv1d(in_features, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.GELU()
        )
        
        # Stage 2: Self Attention Booster
        self.attention = nn.MultiheadAttention(embed_dim=128, num_heads=4, batch_first=True)
        self.layer_norm = nn.LayerNorm(128)
        
        # Stage 3: TCN
        self.tcn = nn.Sequential(
            ResidualBlock(128, 128, dilation=1),
            ResidualBlock(128, 128, dilation=2),
            ResidualBlock(128, 128, dilation=4),
            ResidualBlock(128, 128, dilation=8)
        )
        
        # Output Head
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, out_classes)
        )

    def forward(self, x):
        # x: (B, 34, 14)
        x = self.cnn(x) # (B, 128, 14)
        
        # Attention expects (B, Seq, Features)
        x = x.transpose(1, 2) # (B, 14, 128)
        att_out, _ = self.attention(x, x, x)
        x = self.layer_norm(x + att_out) # Residual add + norm
        
        # TCN expects (B, Features, Seq)
        x = x.transpose(1, 2)
        x = self.tcn(x)
        
        # Take last timestep: (B, 128)
        x = x[:, :, -1]
        
        # Head
        logits = self.head(x)
        return torch.sigmoid(logits)

model = GiaconiaOOSModel().to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3. Training Loop

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Basic training loop (1 epoch for demo due to time constraints, normally 50)
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.submit = optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

model.eval()
with torch.no_grad():
    test_loss = 0
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = model(X_batch)
        test_loss += criterion(preds, y_batch).item()
    print(f"Test Loss: {test_loss/len(test_loader):.4f}")

## 4. Evaluation (F1, Recall, MAE)

In [ ]:
from sklearn.metrics import f1_score, recall_score, mean_absolute_error

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = model(X_batch)
        
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

# Binarize predictions at 0.5 threshold
pred_binary = (all_preds > 0.5).astype(int)

# Flatten for global F1 and Recall
f1 = f1_score(all_targets.flatten(), pred_binary.flatten())
recall = recall_score(all_targets.flatten(), pred_binary.flatten())

# Calculate MAE (hrs) - difference in total predicted OOS hours vs actual
pred_hours = pred_binary.sum(axis=1)
actual_hours = all_targets.sum(axis=1)
mae = mean_absolute_error(actual_hours, pred_hours)

print("=== Final Metrics ===")
print(f"F1-Score: {f1:.4f}")
print(f"Recall:   {recall:.4f} ({(recall*100):.2f}%)")
print(f"MAE (hrs): {mae:.2f}")
